In [9]:
import pandas as pd

# 1. 키워드 정의
keywords = [
    "햄버거", "버거킹", "쉑쉑버거", "롯데리아",
    "샌드위치", "에그드롭", "서브웨이", "홍루이젠",
    "디저트", "노티드", "몽슈슈", "투썸플레이스"
]
df_base = pd.DataFrame({"상세메뉴": keywords})

# 2. 검색수 파일 로딩 및 열 이름 표준화
df_avg = pd.read_excel("네이버_검색_절대숫자.xlsx")
menu_col = next((col for col in df_avg.columns if any(k in col for k in ["상세메뉴", "키워드", "메뉴"])), None)
df_avg = df_avg.rename(columns={menu_col: "상세메뉴"})

# 3. 월별 추세 파일 전처리
df_monthly = pd.read_excel("병합된_상세메뉴_월별추세_결과.xlsx")
df_transposed = df_monthly.set_index("월").transpose().reset_index()
df_transposed.rename(columns={"index": "상세메뉴"}, inplace=True)
month_columns = [col for col in df_transposed.columns if col != "상세메뉴"]
df_transposed[month_columns] = df_transposed[month_columns].apply(pd.to_numeric, errors="coerce").fillna(0)
df_transposed["월평균"] = df_transposed[month_columns].mean(axis=1)

# 4. 유사 매칭 함수
def find_match(name, candidates):
    for c in candidates:
        if isinstance(c, str) and name == c.strip():
            return c
    for c in candidates:
        if isinstance(c, str) and name in c:
            return c
    return None

# 5. 검색수 및 추세 매칭
df_base["상세메뉴_검색수파일"] = df_base["상세메뉴"].apply(lambda x: find_match(x, df_avg["상세메뉴"].astype(str)))
df_base = pd.merge(df_base, df_avg, left_on="상세메뉴_검색수파일", right_on="상세메뉴", how="left", suffixes=("", "_검색수"))

df_base["상세메뉴_추세파일"] = df_base["상세메뉴"].apply(lambda x: find_match(x, df_transposed["상세메뉴"].astype(str)))
df_base = pd.merge(df_base, df_transposed, left_on="상세메뉴_추세파일", right_on="상세메뉴", how="left", suffixes=("", "_추세"))

# 6. 보정 비율 계산
def compute_scaling(row):
    if pd.isna(row["월평균"]) or row["월평균"] < 1e-6:
        return row["총합"] * 0.5 if pd.notna(row["총합"]) else None
    return row["총합"] / row["월평균"] if pd.notna(row["총합"]) else None

df_base["보정비율"] = df_base.apply(compute_scaling, axis=1)

# 7. 보정 추세값 및 비중 계산
for col in month_columns:
    df_base[f"{col}_보정"] = df_base[col] * df_base["보정비율"]
    # 월평균이 없는 경우 대체
    df_base.loc[df_base["보정비율"].isna(), f"{col}_보정"] = df_base.loc[df_base["보정비율"].isna(), "총합"] * 0.5

    # 비중 계산
    total = df_base[f"{col}_보정"].sum()
    df_base[f"{col}_비중"] = df_base[f"{col}_보정"] / total if total > 0 else 0

# 8. 결과 저장
columns_to_export = (
    ["상세메뉴", "상세메뉴_검색수파일", "상세메뉴_추세파일", "총합", "월평균", "보정비율"] +
    [f"{col}_보정" for col in month_columns] +
    [f"{col}_비중" for col in month_columns]
)
df_result = df_base[columns_to_export]
df_result.to_excel("상세메뉴_보정결과_비중포함_출력.xlsx", index=False)

print("✔ 결과 저장 완료: 상세메뉴_보정결과_비중포함_출력.xlsx")

✔ 결과 저장 완료: 상세메뉴_보정결과_비중포함_출력.xlsx
